# Brain tumour MRI classification — experiment notebook

Compare two CNNs trained from scratch with a frozen pretrained ResNet18 classifier using PyTorch. Eight configurations compare learning rates and AdamW/SGD; each was trained for five epochs with batch size 32.

**Recorded result:** original CNN + AdamW, learning rate 0.0003; filtered test accuracy **84.81%**, macro-F1 **0.836**, 1,468 test images. These are previously recorded results, not outputs from a new execution of this cleaned notebook.

**Important limitation:** exact-image filtering happened after training. Original validation images influenced checkpoint selection and LR scheduling. This is a retrospective experiment, not a fully leakage-free benchmark. Patient overlap and near-duplicates were not ruled out. Educational research only; not for diagnosis.

## How to use this notebook

Save this file in the repository's `notebooks` folder. Default **review mode** reads committed reports and figures without loading the dataset, changing model files, or starting training. Run cells from top to bottom.

For an optional evaluation replay, set `RUN_PIPELINE = True` below and point `WORK_DIR` to the original working folder containing `data` and `tuning_checkpoints`. Keep `RUN_TRAINING = False` to reuse checkpoints. All eight `best.pth` and `latest.pth` files are needed. They and the dataset are excluded from GitHub; a clone alone cannot replay training/evaluation. Existing checkpoint file-list guards require the original dataset path representation and ordering.

Only set `RUN_TRAINING = True` to deliberately train/resume the historical experiment. A fresh run may download ResNet18 weights and take hours. For a scientifically corrected future experiment, deduplicate **before** splitting/training and use patient-grouped splits when patient identifiers are available; do not reuse these historical checkpoints.

Dataset source URL and licence still need verification. Model training and dataset contents were not re-executed or independently verified during notebook cleanup.

In [ ]:
from pathlib import Path
import json
import os
from IPython.display import display, Image as DisplayImage

RUN_PIPELINE = False
RUN_TRAINING = False
TOTAL_EPOCHS = 5

# Find the repository from either its root or its notebooks directory.
here = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (here, *here.parents)
     if (p / "predict_brain_tumor.py").is_file()
     and (p / "results").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Open this notebook inside brain-tumor-classifier/notebooks.")

# For the original laptop layout, data lives beside the repository.
# Edit this if your data/checkpoint folder is elsewhere.
WORK_DIR = REPO_ROOT.parent if (REPO_ROOT.parent / "data").is_dir() else REPO_ROOT

# Optional replay outputs go outside the repository and preserve saved reports.
REPLAY_OUTPUT = REPO_ROOT.parent / "brain-tumor-replay-output"
if RUN_TRAINING and not RUN_PIPELINE:
    raise ValueError("Set RUN_PIPELINE=True before enabling training.")
if RUN_PIPELINE:
    for split in ("Training", "Testing"):
        if not (WORK_DIR / "data" / split).is_dir():
            raise FileNotFoundError(f"Missing data/{split} under {WORK_DIR}")
    os.chdir(WORK_DIR)
    REPLAY_OUTPUT.mkdir(exist_ok=True)
print("Mode:", "pipeline replay" if RUN_PIPELINE else "review only — no training")
print("Repository:", REPO_ROOT)


## 1. Review saved results
These reports and figures are read from the repository. Missing artifacts are reported rather than replaced with invented values.

In [ ]:
for name in ("filtered_validation_results.json", "filtered_test_report.json"):
    path = REPO_ROOT / "results" / name
    print("\n" + name)
    if path.exists():
        display(json.loads(path.read_text(encoding="utf-8")))
    else:
        print("Report not found:", path)
for name in ("model_comparison.png", "confusion_matrix.png", "training_curves.png"):
    path = REPO_ROOT / "results" / name
    if path.exists():
        display(DisplayImage(filename=str(path)))


## 2. Dataset, preprocessing and split
The original class-stratified image split reserves 20% of each training class for validation (seed 42). Disjoint indices do not guarantee distinct patients or distinct image content. CNN inputs are RGB, resized to 128 × 128, normalized with mean/std 0.5. ResNet18 uses its pretrained-weight transforms. The cells below are skipped in review mode.

In [ ]:
if RUN_PIPELINE:
    import random
    import torch
    from torchvision import datasets, transforms
    from torch.utils.data import DataLoader, Subset
    from collections import Counter

    # 1. Fixed seed: makes the split repeatable
    SEED = 42
    random.seed(SEED)
    torch.manual_seed(SEED)

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    # 2. Keep the same image preprocessing as your original model
    tf = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(
            [0.5, 0.5, 0.5],
            [0.5, 0.5, 0.5]
        )
    ])

    # 3. Load the existing folders
    full_train_ds = datasets.ImageFolder(
        "data/Training", transform=tf
    )

    test_ds = datasets.ImageFolder(
        "data/Testing", transform=tf
    )

    assert full_train_ds.class_to_idx == test_ds.class_to_idx, (
        "Training and testing class labels do not match."
    )

    class_names = full_train_ds.classes

    # 4. Reserve 20% of EACH class for validation
    rng = random.Random(SEED)
    train_indices = []
    val_indices = []

    for class_id in range(len(class_names)):
        indices = [
            i for i, label in enumerate(full_train_ds.targets)
            if label == class_id
        ]

        rng.shuffle(indices)
        val_count = round(len(indices) * 0.20)

        val_indices.extend(indices[:val_count])
        train_indices.extend(indices[val_count:])

    train_ds = Subset(full_train_ds, train_indices)
    val_ds = Subset(full_train_ds, val_indices)

    # Confirm that no image index belongs to both sets
    assert set(train_indices).isdisjoint(val_indices)

    # 5. Prepare batches of 32 images
    train_dl = DataLoader(
        train_ds,
        batch_size=32,
        shuffle=True,
        num_workers=0,
        generator=torch.Generator().manual_seed(SEED)
    )

    val_dl = DataLoader(
        val_ds,
        batch_size=32,
        shuffle=False,
        num_workers=0
    )

    test_dl = DataLoader(
        test_ds,
        batch_size=32,
        shuffle=False,
        num_workers=0
    )

    # 6. Display the results
    print("Device:", device)
    print("Classes:", class_names)
    print("\nTraining images:", len(train_ds))
    print("Validation images:", len(val_ds))
    print("Testing images:", len(test_ds))

    train_counts = Counter(
        full_train_ds.targets[i] for i in train_indices
    )
    val_counts = Counter(
        full_train_ds.targets[i] for i in val_indices
    )

    print("\nPer-class split:")
    for class_id, name in enumerate(class_names):
        print(
            f"{name}: "
            f"train={train_counts[class_id]}, "
            f"validation={val_counts[class_id]}"
        )

## 3. Search space and model architectures
Learning rate controls update size. AdamW and SGD update trainable weights; weight decay regularizes them. Dropout is 0.5 in the custom CNNs. ResNet18 uses a frozen backbone with a trainable linear classifier, keeping backbone BatchNorm statistics fixed. This is feature extraction, not full-backbone fine-tuning.

In [ ]:
if RUN_PIPELINE:
    # Small initial search: 3 architectures × 2 learning rates
    # plus an optimizer comparison on the original CNN.

    experiments = [
        {
            "name": "original_adamw_lr0001",
            "architecture": "original_cnn",
            "optimizer": "AdamW",
            "lr": 1e-4,
        },
        {
            "name": "original_adamw_lr0003",
            "architecture": "original_cnn",
            "optimizer": "AdamW",
            "lr": 3e-4,
        },
        {
            "name": "original_sgd_lr001",
            "architecture": "original_cnn",
            "optimizer": "SGD",
            "lr": 1e-2,
        },
        {
            "name": "original_sgd_lr0001",
            "architecture": "original_cnn",
            "optimizer": "SGD",
            "lr": 1e-3,
        },
        {
            "name": "small_adamw_lr0001",
            "architecture": "small_cnn",
            "optimizer": "AdamW",
            "lr": 1e-4,
        },
        {
            "name": "small_adamw_lr0003",
            "architecture": "small_cnn",
            "optimizer": "AdamW",
            "lr": 3e-4,
        },
        {
            "name": "resnet18_adamw_lr0001",
            "architecture": "resnet18",
            "optimizer": "AdamW",
            "lr": 1e-4,
        },
        {
            "name": "resnet18_adamw_lr0003",
            "architecture": "resnet18",
            "optimizer": "AdamW",
            "lr": 3e-4,
        },
    ]

    # Shared settings
    BATCH_SIZE = 32
    DROPOUT = 0.5
    WEIGHT_DECAY = 1e-4

    # Scheduler settings: reduce LR when validation loss stalls
    LR_FACTOR = 0.5
    LR_PATIENCE = 1

    print("Experiment configurations prepared:", len(experiments))
    print("No training has started.")

    for config in experiments:
        print(
            f"{config['architecture']:12s} | "
            f"{config['optimizer']:5s} | "
            f"learning rate = {config['lr']}"
        )

In [ ]:
if RUN_PIPELINE:
    from torch import nn

    def build_model(dropout=0.5):
        model = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, len(class_names))
        )

        return model.to(device)


    loss_fn = nn.CrossEntropyLoss()


In [ ]:
if RUN_PIPELINE:
    from torchvision.models import resnet18, ResNet18_Weights

    # Pretrained weights and their matching image preprocessing
    RESNET_WEIGHTS = ResNet18_Weights.DEFAULT
    resnet_tf = RESNET_WEIGHTS.transforms()


    # Architecture 2: smaller CNN, trained from scratch
    def build_small_cnn(dropout=0.5):
        return nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),

            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, len(class_names))
        ).to(device)


    # Architecture 3: frozen pretrained ResNet18 + new classifier
    class FrozenResNet18(nn.Module):
        def __init__(self, num_classes, pretrained=True):
            super().__init__()

            self.backbone = resnet18(
                weights=RESNET_WEIGHTS if pretrained else None
            )

            feature_count = self.backbone.fc.in_features
            self.backbone.fc = nn.Identity()

            # Freeze all pretrained parameters
            for parameter in self.backbone.parameters():
                parameter.requires_grad = False

            self.classifier = nn.Linear(feature_count, num_classes)
            self.backbone.eval()

        def train(self, mode=True):
            super().train(mode)

            # Keep BatchNorm statistics in the frozen backbone unchanged
            self.backbone.eval()
            return self

        def forward(self, x):
            with torch.no_grad():
                features = self.backbone(x)

            return self.classifier(features)


    # Choose a model using the architecture name in our experiments
    def create_model(architecture, pretrained=True):
        if architecture == "original_cnn":
            return build_model(dropout=DROPOUT)

        if architecture == "small_cnn":
            return build_small_cnn(dropout=DROPOUT)

        if architecture == "resnet18":
            return FrozenResNet18(
                num_classes=len(class_names),
                pretrained=pretrained
            ).to(device)

        raise ValueError(f"Unknown architecture: {architecture}")




## 4. Optimizers, learning-rate scheduler and checkpointing
ReduceLROnPlateau monitors original validation loss with factor 0.5 and patience 1. A configured scheduler does not imply LR actually decreased during five epochs. `best.pth` stores the minimum original-validation-loss checkpoint; `latest.pth` supports epoch-boundary resume with optimizer, scheduler and random states. Exact reproducibility across devices/software is not guaranteed.

In [ ]:
if RUN_PIPELINE:
    from torch import optim

    # 1. Give each architecture the correct preprocessing.
    # All experiments still use the SAME image split.
    def make_loaders(architecture):
        transform = resnet_tf if architecture == "resnet18" else tf

        dataset = datasets.ImageFolder(
            "data/Training",
            transform=transform
        )

        # Ensure our existing split indices still refer to the same files
        assert dataset.samples == full_train_ds.samples

        training_subset = Subset(dataset, train_indices)
        validation_subset = Subset(dataset, val_indices)

        # Reset image-shuffling seed for each experiment
        generator = torch.Generator().manual_seed(SEED)

        training_loader = DataLoader(
            training_subset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=0,
            generator=generator
        )

        validation_loader = DataLoader(
            validation_subset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=0
        )

        return training_loader, validation_loader


    # 2. Create the optimizer specified by an experiment
    def make_optimizer(model, config):
        trainable_parameters = [
            p for p in model.parameters() if p.requires_grad
        ]

        if config["optimizer"] == "AdamW":
            return optim.AdamW(
                trainable_parameters,
                lr=config["lr"],
                weight_decay=WEIGHT_DECAY
            )

        if config["optimizer"] == "SGD":
            return optim.SGD(
                trainable_parameters,
                lr=config["lr"],
                momentum=0.9,
                weight_decay=WEIGHT_DECAY
            )

        raise ValueError("Unknown optimizer: " + config["optimizer"])


    # 3. Reduce LR when validation loss stops improving
    def make_scheduler(optimizer):
        return optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=LR_FACTOR,
            patience=LR_PATIENCE,
            threshold=1e-4,
            min_lr=1e-6
        )




In [ ]:
if RUN_PIPELINE:
    import time
    import os
    from pathlib import Path

    CHECKPOINT_DIR = Path("tuning_checkpoints")
    CHECKPOINT_DIR.mkdir(exist_ok=True)


    # Run one complete training OR validation epoch
    def run_epoch(model, loader, optimizer=None):
        training = optimizer is not None
        model.train(training)

        total_loss = 0.0
        correct = 0
        total_images = 0
        start_time = time.perf_counter()

        for batch_number, (images, labels) in enumerate(loader, start=1):
            images = images.to(device)
            labels = labels.to(device)

            if training:
                optimizer.zero_grad(set_to_none=True)

            # Calculate gradients only during training
            with torch.set_grad_enabled(training):
                predictions = model(images)
                loss = loss_fn(predictions, labels)

                if not torch.isfinite(loss):
                    raise RuntimeError("Loss became NaN or infinite.")

                if training:
                    loss.backward()
                    optimizer.step()

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            correct += (
                predictions.argmax(dim=1) == labels
            ).sum().item()
            total_images += batch_size

            if batch_number == 1 or batch_number % 20 == 0:
                stage = "Training" if training else "Validation"
                print(
                    f"  {stage}: batch {batch_number}/{len(loader)}",
                    flush=True
                )

        return {
            "loss": total_loss / total_images,
            "accuracy": 100.0 * correct / total_images,
            "seconds": time.perf_counter() - start_time
        }


    # Save enough information to resume after a completed epoch
    def save_checkpoint(
        path, model, optimizer, scheduler,
        completed_epochs, config, history,
        best_val_loss, training_loader
    ):
        checkpoint = {
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),

            # Number of FULLY completed epochs, rather than epoch index
            "completed_epochs": completed_epochs,

            "config": dict(config),
            "history": list(history),
            "best_val_loss": best_val_loss,
            "class_names": list(class_names),

            "train_indices": list(train_indices),
            "val_indices": list(val_indices),
            "training_samples": list(full_train_ds.samples),

            "settings": {
                "seed": SEED,
                "batch_size": BATCH_SIZE,
                "dropout": DROPOUT,
                "weight_decay": WEIGHT_DECAY,
                "lr_factor": LR_FACTOR,
                "lr_patience": LR_PATIENCE
            },

            # Preserve random states for resuming training
            "torch_rng_state": torch.get_rng_state(),
            "python_rng_state": random.getstate(),
            "loader_rng_state": training_loader.generator.get_state(),
            "cuda_rng_state": (
                torch.cuda.get_rng_state_all()
                if torch.cuda.is_available() else None
            )
        }

        # Finish writing a temporary file before replacing the checkpoint
        path = Path(path)
        temporary_path = path.with_suffix(path.suffix + ".tmp")
        torch.save(checkpoint, temporary_path)
        os.replace(temporary_path, path)


    print("Training and validation functions ready.")
    print("Checkpoint-saving function ready.")
    print("Checkpoint folder:", CHECKPOINT_DIR.resolve())
    print("No training started yet.")

In [ ]:
if RUN_PIPELINE:
    def train_experiment(config, total_epochs=5):
        config = dict(config)

        run_dir = CHECKPOINT_DIR / config["name"]
        run_dir.mkdir(parents=True, exist_ok=True)

        latest_path = run_dir / "latest.pth"
        best_path = run_dir / "best.pth"

        settings = {
            "seed": SEED,
            "batch_size": BATCH_SIZE,
            "dropout": DROPOUT,
            "weight_decay": WEIGHT_DECAY,
            "lr_factor": LR_FACTOR,
            "lr_patience": LR_PATIENCE
        }

        random.seed(SEED)
        torch.manual_seed(SEED)

        training_loader, validation_loader = make_loaders(
            config["architecture"]
        )

        # When resuming, pretrained weights come from the checkpoint
        model = create_model(
            config["architecture"],
            pretrained=not latest_path.exists()
        )
        optimizer = make_optimizer(model, config)
        scheduler = make_scheduler(optimizer)

        completed_epochs = 0
        best_val_loss = float("inf")
        history = []

        if latest_path.exists():
            checkpoint = torch.load(
                latest_path,
                map_location="cpu",
                weights_only=True
            )

            # Prevent mixing different settings or data splits
            assert checkpoint["config"] == config, "Config changed."
            assert checkpoint["settings"] == settings, "Settings changed."
            assert checkpoint["class_names"] == list(class_names)
            assert checkpoint["train_indices"] == list(train_indices)
            assert checkpoint["val_indices"] == list(val_indices)
            assert checkpoint["training_samples"] == list(
                full_train_ds.samples
            ), "Dataset file list changed."

            model.load_state_dict(checkpoint["model_state"])
            optimizer.load_state_dict(checkpoint["optimizer_state"])
            scheduler.load_state_dict(checkpoint["scheduler_state"])

            completed_epochs = checkpoint["completed_epochs"]
            best_val_loss = checkpoint["best_val_loss"]
            history = checkpoint["history"]

            # Restore randomness AFTER constructing the model
            torch.set_rng_state(checkpoint["torch_rng_state"])
            random.setstate(checkpoint["python_rng_state"])
            training_loader.generator.set_state(
                checkpoint["loader_rng_state"]
            )

            if (
                torch.cuda.is_available()
                and checkpoint["cuda_rng_state"] is not None
            ):
                torch.cuda.set_rng_state_all(
                    checkpoint["cuda_rng_state"]
                )

            del checkpoint

            print(
                f"Loaded checkpoint: {completed_epochs} epochs completed.",
                flush=True
            )

        elif best_path.exists():
            raise RuntimeError(
                "Found best.pth without latest.pth. "
                "Check the files before starting again."
            )

        if completed_epochs >= total_epochs:
            print(f"Already completed {total_epochs} or more epochs.")
            return history

        for epoch_number in range(completed_epochs + 1, total_epochs + 1):
            print(
                f"\n{config['name']} | Epoch {epoch_number}/{total_epochs}",
                flush=True
            )

            epoch_start = time.perf_counter()
            lr_used = optimizer.param_groups[0]["lr"]

            train_metrics = run_epoch(
                model, training_loader, optimizer
            )
            val_metrics = run_epoch(
                model, validation_loader
            )

            scheduler.step(val_metrics["loss"])
            next_lr = optimizer.param_groups[0]["lr"]

            improved = val_metrics["loss"] < best_val_loss
            if improved:
                best_val_loss = val_metrics["loss"]

            history.append({
                "epoch": epoch_number,
                "train_loss": train_metrics["loss"],
                "train_accuracy": train_metrics["accuracy"],
                "val_loss": val_metrics["loss"],
                "val_accuracy": val_metrics["accuracy"],
                "lr_used": lr_used,
                "next_lr": next_lr,
                "seconds": time.perf_counter() - epoch_start
            })

            # Save the best model only when validation loss improves
            paths_to_save = [best_path, latest_path] if improved else [
                latest_path
            ]

            for path in paths_to_save:
                save_checkpoint(
                    path=path,
                    model=model,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    completed_epochs=epoch_number,
                    config=config,
                    history=history,
                    best_val_loss=best_val_loss,
                    training_loader=training_loader
                )

            print(
                f"Train accuracy: {train_metrics['accuracy']:.2f}% | "
                f"Validation accuracy: {val_metrics['accuracy']:.2f}%",
                flush=True
            )
            print(
                f"Validation loss: {val_metrics['loss']:.4f} | "
                f"LR used: {lr_used:g} | Next LR: {next_lr:g}",
                flush=True
            )
            print(
                "Latest checkpoint saved."
                + (" Best checkpoint updated." if improved else ""),
                flush=True
            )

        return history


    print("Training function ready. Supports automatic resume.")

## 5. Optional training or checkpoint availability check
There is only one training launch. With training disabled, this cell verifies all required checkpoints before evaluation. Existing checkpoints are not overwritten by evaluation.

In [ ]:
if RUN_PIPELINE:
    if RUN_TRAINING:
        all_histories = {}
        for number, config in enumerate(experiments, 1):
            print(f"Experiment {number}/{len(experiments)}: {config['name']}")
            all_histories[config["name"]] = train_experiment(config, TOTAL_EPOCHS)
    else:
        missing = [str(CHECKPOINT_DIR / c["name"] / filename)
                   for c in experiments for filename in ("best.pth", "latest.pth")
                   if not (CHECKPOINT_DIR / c["name"] / filename).is_file()]
        if missing:
            raise FileNotFoundError("Missing checkpoints; restore original files:\n" + "\n".join(missing))
        print("All checkpoints found; training skipped.")


## 6. Retrospective exact-image audit
SHA-256 hashes decoded RGB pixel bytes and dimensions. Recorded overlaps were 47 train/validation, 85 train/test and 26 validation/test shared image groups; these pairwise counts are not additive. Filtering retains one validation image per hash absent from training, and one test image per hash absent from both original development splits. No source image is deleted. This does not detect all near-duplicates or identify patient overlap.

In [ ]:
if RUN_PIPELINE:
    import hashlib
    from PIL import Image
    from collections import defaultdict
    from itertools import combinations

    split_samples = {
        "Training": [full_train_ds.samples[i] for i in train_indices],
        "Validation": [full_train_ds.samples[i] for i in val_indices],
        "Testing": test_ds.samples,
    }

    image_hashes = {}

    for split_name, samples in split_samples.items():
        hashes = defaultdict(list)

        for path, label in samples:
            with Image.open(path) as image:
                image = image.convert("RGB")
                digest = hashlib.sha256(
                    str(image.size).encode() + image.tobytes()
                ).hexdigest()

            hashes[digest].append((path, label))

        image_hashes[split_name] = hashes
        print(f"{split_name}: checked {len(samples)} images.")

    duplicate_overlaps = {}

    for first, second in combinations(split_samples, 2):
        shared = set(image_hashes[first]) & set(image_hashes[second])
        duplicate_overlaps[(first, second)] = shared

        print(
            f"\n{first} / {second}: "
            f"{len(shared)} shared exact-image groups"
        )

        for digest in sorted(shared)[:3]:
            print("  ", image_hashes[first][digest][0][0])
            print("  ", image_hashes[second][digest][0][0])

    print("\nDuplicate check complete.")

In [ ]:
if RUN_PIPELINE:
    from torch.utils.data import Subset
    from collections import Counter, defaultdict

    # Check whether identical images have conflicting class labels.
    labels_by_hash = defaultdict(set)

    for hashes in image_hashes.values():
        for digest, records in hashes.items():
            labels_by_hash[digest].update(label for _, label in records)

    conflicts = [
        digest for digest, labels in labels_by_hash.items()
        if len(labels) > 1
    ]

    if conflicts:
        raise ValueError(
            f"{len(conflicts)} identical-image groups have conflicting labels. "
            "Resolve label conflicts before evaluation."
        )

    train_hashes = set(image_hashes["Training"])
    original_val_hashes = set(image_hashes["Validation"])

    # Keep one copy of each validation image not seen during training.
    clean_val_paths = {
        records[0][0]
        for digest, records in image_hashes["Validation"].items()
        if digest not in train_hashes
    }

    # Exclude anything present in either original development split.
    clean_test_paths = {
        records[0][0]
        for digest, records in image_hashes["Testing"].items()
        if digest not in (train_hashes | original_val_hashes)
    }

    clean_val_indices = [
        i for i in val_indices
        if full_train_ds.samples[i][0] in clean_val_paths
    ]

    clean_test_indices = [
        i for i, (path, _) in enumerate(test_ds.samples)
        if path in clean_test_paths
    ]

    clean_val_ds = Subset(full_train_ds, clean_val_indices)
    clean_test_ds = Subset(test_ds, clean_test_indices)

    for name, dataset in [
        ("Filtered validation", clean_val_ds),
        ("Filtered testing", clean_test_ds),
    ]:
        counts = Counter(
            dataset.dataset.samples[i][1] for i in dataset.indices
        )
        print(f"\n{name}: {len(dataset)} images")
        for class_id, class_name in enumerate(class_names):
            print(f"  {class_name}: {counts[class_id]}")
        assert all(counts[i] > 0 for i in range(len(class_names)))

    print("\nEvaluation subsets ready. No files deleted.")

## 7. Compare saved models on filtered validation
The recorded filtered validation set contains 1,064 images. Select by minimum filtered validation loss, never by test accuracy. Historical checkpoint epochs were still selected on original validation.

In [ ]:
if RUN_PIPELINE:
    import torch
    from pathlib import Path
    from torchvision import datasets
    from torch.utils.data import DataLoader, Subset

    filtered_results = []

    for number, config in enumerate(experiments, start=1):
        print(f"\nEvaluating {number}/8: {config['name']}", flush=True)

        # Use the preprocessing required by each architecture.
        transform = resnet_tf if config["architecture"] == "resnet18" else tf
        dataset = datasets.ImageFolder("data/Training", transform=transform)
        assert dataset.samples == full_train_ds.samples

        loader = DataLoader(
            Subset(dataset, clean_val_indices),
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=0
        )

        path = Path("tuning_checkpoints") / config["name"] / "best.pth"
        checkpoint = torch.load(path, map_location="cpu", weights_only=True)
        assert checkpoint["class_names"] == list(class_names)
        assert checkpoint["config"] == config
        assert checkpoint["training_samples"] == list(full_train_ds.samples)
        assert checkpoint["train_indices"] == list(train_indices)
        assert checkpoint["val_indices"] == list(val_indices)

        candidate = create_model(config["architecture"], pretrained=False)
        candidate.load_state_dict(checkpoint["model_state"])

        metrics = run_epoch(candidate, loader)

        filtered_results.append({
            "name": config["name"],
            "val_loss": metrics["loss"],
            "val_accuracy": metrics["accuracy"],
            "checkpoint_epoch": checkpoint["completed_epochs"]
        })

        del candidate, checkpoint

    filtered_results.sort(key=lambda result: result["val_loss"])

    print("\nFILTERED VALIDATION RESULTS")
    print(f"{'Experiment':<30} {'Loss':>10} {'Accuracy':>12}")

    for result in filtered_results:
        print(
            f"{result['name']:<30} "
            f"{result['val_loss']:>10.4f} "
            f"{result['val_accuracy']:>11.2f}%"
        )

    best_name = filtered_results[0]["name"]
    best_config = next(
        config for config in experiments if config["name"] == best_name
    )

    print("\nSelected experiment:", best_name)

## 8. Final test evaluation and metrics
Report accuracy, per-class precision/recall/F1, macro-F1 and the confusion matrix. Macro-F1 gives equal weight to each class. Replay reports go to a separate folder so committed results stay intact; use test results for reporting, not repeated model selection.

In [ ]:
if RUN_PIPELINE:
    import json
    import csv
    import torch
    from pathlib import Path
    from torchvision import datasets
    from torch.utils.data import DataLoader, Subset

    results_dir = REPLAY_OUTPUT / "results"
    results_dir.mkdir(parents=True, exist_ok=True)

    # Save the completed validation comparison.
    (results_dir / "filtered_validation_results.json").write_text(
        json.dumps(filtered_results, indent=2), encoding="utf-8"
    )

    # Load the selected model.
    best_path = Path("tuning_checkpoints") / best_name / "best.pth"
    best_checkpoint = torch.load(
        best_path, map_location="cpu", weights_only=True
    )
    assert best_checkpoint["class_names"] == list(class_names)

    best_model = create_model(best_config["architecture"], pretrained=False)
    best_model.load_state_dict(best_checkpoint["model_state"])
    best_model.eval()

    # Match preprocessing to the selected architecture.
    test_transform = (
        resnet_tf if best_config["architecture"] == "resnet18" else tf
    )
    evaluation_dataset = datasets.ImageFolder(
        "data/Testing", transform=test_transform
    )
    assert evaluation_dataset.samples == test_ds.samples

    evaluation_loader = DataLoader(
        Subset(evaluation_dataset, clean_test_indices),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=0
    )

    y_true, y_pred = [], []
    test_loss_sum = 0.0

    with torch.inference_mode():
        for batch_number, (images, labels) in enumerate(
            evaluation_loader, start=1
        ):
            outputs = best_model(images.to(device))
            test_loss_sum += (
                loss_fn(outputs, labels.to(device)).item() * labels.size(0)
            )
            y_true.extend(labels.tolist())
            y_pred.extend(outputs.argmax(dim=1).cpu().tolist())

            if batch_number == 1 or batch_number % 10 == 0:
                print(
                    f"Testing batch {batch_number}/{len(evaluation_loader)}",
                    flush=True
                )

    # Rows = true classes; columns = predicted classes.
    cm = torch.zeros(
        (len(class_names), len(class_names)), dtype=torch.int64
    )
    for actual, predicted in zip(y_true, y_pred):
        cm[actual, predicted] += 1

    tp = cm.diag().double()
    support = cm.sum(dim=1)
    precision = tp / cm.sum(dim=0).clamp(min=1)
    recall = tp / support.clamp(min=1)
    f1 = 2 * precision * recall / (precision + recall).clamp(min=1e-12)
    test_accuracy = 100 * tp.sum().item() / len(y_true)

    print(f"\nFILTERED TEST ACCURACY: {test_accuracy:.2f}%")
    print(f"{'Class':<15} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Images':>8}")

    per_class = {}
    for i, name in enumerate(class_names):
        per_class[name] = {
            "precision": precision[i].item(),
            "recall": recall[i].item(),
            "f1": f1[i].item(),
            "support": support[i].item()
        }
        print(
            f"{name:<15} {precision[i]:>10.3f} "
            f"{recall[i]:>10.3f} {f1[i]:>10.3f} {support[i]:>8}"
        )

    print(f"\nMacro F1: {f1.mean().item():.3f}")

    test_report = {
        "experiment": best_name,
        "test_images": len(y_true),
        "accuracy_percent": test_accuracy,
        "loss": test_loss_sum / len(y_true),
        "macro_f1": f1.mean().item(),
        "per_class": per_class,
        "class_names": list(class_names),
        "confusion_matrix": cm.tolist(),
        "limitation": (
            "Exact duplicates filtered after training. Original validation "
            "influenced training decisions. Patient overlap and near-duplicates "
            "have not been ruled out."
        )
    }
    (results_dir / "filtered_test_report.json").write_text(
        json.dumps(test_report, indent=2), encoding="utf-8"
    )

    with (results_dir / "test_predictions.csv").open(
        "w", newline="", encoding="utf-8"
    ) as file:
        writer = csv.writer(file)
        writer.writerow(["image", "true_class", "predicted_class"])
        for index, actual, predicted in zip(
            clean_test_indices, y_true, y_pred
        ):
            writer.writerow([
                evaluation_dataset.samples[index][0],
                class_names[actual],
                class_names[predicted]
            ])

    print("\nResults saved in:", results_dir.resolve())

## 9. Diagnostic figures
Training curves use original validation metrics. The comparison chart uses retrospectively filtered validation loss. Dataset image montages are omitted from this shareable notebook pending source/licence verification.

In [ ]:
if RUN_PIPELINE:
    import matplotlib.pyplot as plt
    import torch

    # 1. Confusion matrix
    matrix = cm.cpu().numpy()

    fig, ax = plt.subplots(figsize=(7, 6))
    plot = ax.imshow(matrix, cmap="Blues")
    fig.colorbar(plot, ax=ax)

    ax.set(
        xticks=range(len(class_names)),
        yticks=range(len(class_names)),
        xticklabels=class_names,
        yticklabels=class_names,
        xlabel="Predicted class",
        ylabel="Actual class",
        title="Confusion matrix — filtered test set"
    )
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")

    for i in range(len(class_names)):
        for j in range(len(class_names)):
            ax.text(
                j, i, str(matrix[i, j]),
                ha="center", va="center",
                color="white" if matrix[i, j] > matrix.max() / 2 else "black"
            )

    fig.tight_layout()
    fig.savefig(results_dir / "confusion_matrix.png", dpi=200)
    plt.show()

    # 2. Full training history of the selected experiment
    latest_checkpoint = torch.load(
        best_path.parent / "latest.pth",
        map_location="cpu",
        weights_only=True
    )
    history = latest_checkpoint["history"]
    epochs = [row["epoch"] for row in history]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(
        epochs, [row["train_loss"] for row in history],
        "o-", label="Training"
    )
    axes[0].plot(
        epochs, [row["val_loss"] for row in history],
        "o-", label="Original validation"
    )
    axes[0].set(xlabel="Epoch", ylabel="Loss", title="Loss curves")

    axes[1].plot(
        epochs, [row["train_accuracy"] for row in history],
        "o-", label="Training"
    )
    axes[1].plot(
        epochs, [row["val_accuracy"] for row in history],
        "o-", label="Original validation"
    )
    axes[1].set(
        xlabel="Epoch", ylabel="Accuracy (%)", title="Accuracy curves"
    )

    for ax in axes:
        ax.set_xticks(epochs)
        ax.legend()
        ax.grid(alpha=0.3)

    fig.suptitle(
        "Recorded training history — validation duplicates were filtered later",
        fontsize=11
    )
    fig.tight_layout()
    fig.savefig(results_dir / "training_curves.png", dpi=200)
    plt.show()

    print("Both plots saved in:", results_dir.resolve())

In [ ]:
if RUN_PIPELINE:

    ordered = sorted(filtered_results, key=lambda row: row["val_loss"])

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.barh(
        [row["name"] for row in ordered],
        [row["val_loss"] for row in ordered],
        color=["seagreen"] + ["steelblue"] * (len(ordered) - 1)
    )
    ax.invert_yaxis()
    ax.set(
        xlabel="Filtered validation loss — lower is better",
        title="Comparison of eight saved models"
    )
    fig.tight_layout()
    fig.savefig(results_dir / "model_comparison.png", dpi=200)
    plt.show()

    print("Model comparison saved.")

## 10. Export the selected original CNN and verify reload
The existing standalone prediction script supports the original CNN. This cell explicitly stops if another architecture wins; such a model needs a matching exporter/inference script. Exported weights go to the separate replay folder. Softmax scores are not calibrated clinical probabilities.

In [ ]:
if RUN_PIPELINE:
    from pathlib import Path
    import torch

    export_dir = REPLAY_OUTPUT / "exported_model"
    export_dir.mkdir(exist_ok=True)
    export_path = export_dir / "brain_tumor_model.pth"

    assert best_config["architecture"] == "original_cnn"

    # Save weights and the information needed for prediction.
    export_data = {
        "model_state": {
            name: value.detach().cpu().clone()
            for name, value in best_model.state_dict().items()
        },
        "architecture": best_config["architecture"],
        "experiment": best_name,
        "class_names": list(class_names),
        "dropout": DROPOUT,
        "preprocessing": {
            "color_mode": "RGB",
            "image_size": [128, 128],
            "mean": [0.5, 0.5, 0.5],
            "std": [0.5, 0.5, 0.5]
        },
        "evaluation": test_report,
        "intended_use": "Educational research; not for clinical diagnosis."
    }

    torch.save(export_data, export_path)

    # Reload and check that its outputs match the evaluated model.
    saved = torch.load(
        export_path, map_location="cpu", weights_only=True
    )
    reloaded_model = create_model(
        saved["architecture"], pretrained=False
    )
    reloaded_model.load_state_dict(saved["model_state"])
    reloaded_model.eval()
    best_model.eval()

    sample_images, _ = next(iter(evaluation_loader))
    sample_images = sample_images[:2].to(device)

    with torch.inference_mode():
        original_output = best_model(sample_images)
        reloaded_output = reloaded_model(sample_images)

    torch.testing.assert_close(original_output, reloaded_output)
    del reloaded_model

    print("MODEL EXPORTED AND RELOAD CHECK PASSED")
    print("Saved at:", export_path.resolve())
    print(f"File size: {export_path.stat().st_size / (1024**2):.1f} MB")

## 11. Verify standalone inference
Use the existing repository script; do not generate or overwrite source files inside this notebook. Matching one prediction verifies consistency, not clinical validity or correctness on every image.

In [ ]:
if RUN_PIPELINE:
    import subprocess
    import sys
    example_path = evaluation_dataset.samples[clean_test_indices[0]][0]
    completed = subprocess.run(
        [sys.executable, str(REPO_ROOT / "predict_brain_tumor.py"),
         example_path, "--model", str(export_path)],
        capture_output=True, text=True, check=True,
    )
    prediction = json.loads(completed.stdout)
    assert prediction["predicted_class"] == class_names[y_pred[0]]
    print(completed.stdout)
    print("Standalone inference consistency check passed.")


## Interpretation and next research steps

- The recorded best model is the original CNN with AdamW at 0.0003, not ResNet18. Transfer learning was evaluated, not assumed superior.
- Eight short runs form a limited manual hyperparameter search, not exhaustive or Bayesian optimization.
- Exact duplicate filtering was retrospective. The existing scores must retain that qualification.
- A future evaluation should use verified dataset provenance, pretraining duplicate control, patient-level separation and an untouched external test set when available.
- Preserve the original checkpoint directory locally. It is not the same as the exported inference model stored on GitHub.

Tutorial starting point: https://www.youtube.com/watch?v=mGJ6OZmR-Xk . Dataset URL and reuse terms: pending verification. This cleaned notebook documents and organizes the historical workflow; it does not claim fresh training or validation.